# CarDD 车损实例分割 · YOLO 分割模型训练

CA6129 课程项目 · Computer Vision & Applications

本 notebook 只负责 **YOLO** 这一支，跟 Mask R-CNN 那份 notebook 完全独立——两个可以分别开两次 Colab 会话跑，互不影响，谁先跑、隔多久跑都没关系。训练结果存到 Drive 后，去 `03_Compare_results.ipynb` 拼对比表格。


## 0 · 环境准备（本 notebook 独立完整，不依赖其他 notebook 跑过什么）

跟其它 notebook 完全独立：新开一个 Colab 会话，只运行这一份文件就能跑完这个模型的训练 + 评测 + 存结果。不需要提前跑过 YOLO 那份，也不需要跑过对比表格那份。

**Drive 目录结构（跟之前一样）：**

```
MyDrive/
  CarDD_project/
    cardd_code/            <- 代码文件夹（不是 zip）
    CarDD_release/         <- 官方 CarDD COCO 数据
    manifests/              <- (自动创建) manifest 缓存，生成一次后所有 notebook 和会话都复用
    colab_results/          <- (自动创建) 训练结果、权重、对比图都存这里
```

**运行时：Runtime → Change runtime type → GPU (T4 或更高)**


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ====== 路径配置：按你 Drive 里的实际目录改这几行 ======
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/CarDD_project'
CODE_DIRNAME = 'cardd_code'
CARDD_RAW_DIRNAME = 'CarDD_release'
# ======================================================

import sys, os, json, shutil
from pathlib import Path

DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
CODE_DIR = DRIVE_PROJECT_DIR / CODE_DIRNAME
CARDD_DRIVE_RAW_DIR = DRIVE_PROJECT_DIR / CARDD_RAW_DIRNAME
RESULTS_DRIVE_DIR = DRIVE_PROJECT_DIR / 'colab_results'
MANIFEST_DRIVE_DIR = DRIVE_PROJECT_DIR / 'manifests'
RESULTS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

assert CODE_DIR.exists(), f'找不到代码文件夹: {CODE_DIR}'
assert CARDD_DRIVE_RAW_DIR.exists(), f'找不到 CarDD 数据: {CARDD_DRIVE_RAW_DIR}'

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print('代码文件夹:', CODE_DIR)
print('结果保存位置:', RESULTS_DRIVE_DIR)
print('manifest 缓存位置:', MANIFEST_DRIVE_DIR)


代码文件夹: /content/drive/MyDrive/CarDD_project/cardd_code
结果保存位置: /content/drive/MyDrive/CarDD_project/colab_results
manifest 缓存位置: /content/drive/MyDrive/CarDD_project/manifests


In [3]:
!pip install -q -r "{CODE_DIR}/requirements.txt"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.2 MB/s eta 0:00:00


In [4]:
import torch, os
print('PyTorch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    YOLO_DEVICE = '0'
    TORCH_DEVICE = 'cuda'
else:
    print('警告：没有检测到 GPU，训练会很慢。去 Runtime -> Change runtime type 开 GPU。')
    YOLO_DEVICE = 'cpu'
    TORCH_DEVICE = 'cpu'

# Colab 免费版通常只有 2 个 vCPU，dataloader workers 设太多反而互相抢核；
# 按实际核数留一个给主进程，作为下面训练调用里 num_workers/workers 的默认值
NUM_WORKERS = max(1, (os.cpu_count() or 2) - 1)
print('CPU 核数:', os.cpu_count(), '-> dataloader workers 用:', NUM_WORKERS)


PyTorch: 2.11.0+cu128
CUDA 可用: True
GPU: Tesla T4
CPU 核数: 2 -> dataloader workers 用: 1


In [5]:
# 把 CarDD 原始数据从 Drive 复制一份到 Colab 本地磁盘
# Drive 挂载盘对大量小文件读取很慢，训练前拷到本地能省下不少时间
CARDD_LOCAL_RAW_DIR = Path('/content/data/raw/CarDD_release')
if not CARDD_LOCAL_RAW_DIR.exists():
    CARDD_LOCAL_RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
    print('正在从 Drive 复制数据到本地磁盘，几千张图片，可能需要几分钟...')
    !cp -r "{CARDD_DRIVE_RAW_DIR}" "{CARDD_LOCAL_RAW_DIR}"
    print('复制完成')
else:
    print('本地已有数据副本，跳过复制')


正在从 Drive 复制数据到本地磁盘，几千张图片，可能需要几分钟...
复制完成


In [6]:
# manifest 现在会先看 Drive 里有没有已经生成过的缓存，有就直接复制到本地用，
# 不用每个新 session 都重新生成一次（虽然生成本来就很快，只读 JSON 标注不碰图片，
# 但既然能完全复用就没必要每次都重跑）。第一次运行会自动把生成结果也存回 Drive，
# 之后不管是这份 notebook 还是 YOLO 那份 notebook，新开多少次 session 都直接复用。
from dataset.prepare_cardd import build_manifest

MANIFEST_PATH = Path('/content/data/manifests/cardd_manifest.json')
MANIFEST_DRIVE_PATH = MANIFEST_DRIVE_DIR / 'cardd_manifest.json'
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

def ensure_manifest():
    if MANIFEST_PATH.exists():
        print('本地已有 manifest，跳过:', MANIFEST_PATH)
    elif MANIFEST_DRIVE_PATH.exists():
        print('Drive 上已有 manifest 缓存，直接复制到本地（跨 session 复用）:', MANIFEST_DRIVE_PATH)
        shutil.copy(MANIFEST_DRIVE_PATH, MANIFEST_PATH)
    else:
        print('Drive 和本地都没有，重新生成 manifest...')
        manifest = build_manifest(CARDD_LOCAL_RAW_DIR, seed=42)
        MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
        shutil.copy(MANIFEST_PATH, MANIFEST_DRIVE_PATH)
        print('已存回 Drive，之后的 session 都会直接复用:', MANIFEST_DRIVE_PATH)
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    print('数据集划分:', manifest['summary']['images'])
    print('标注数量:', manifest['summary']['annotations'])
    print('类别:', manifest['classes'])
    return manifest

manifest_preview = ensure_manifest()


Drive 上已有 manifest 缓存，直接复制到本地（跨 session 复用）: /content/drive/MyDrive/CarDD_project/manifests/cardd_manifest.json
数据集划分: {'train': 2816, 'validation': 810, 'test': 374}
标注数量: {'train': 6211, 'validation': 1744, 'test': 785}
类别: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']


---
## 1 · 转换成 YOLO 格式数据

YOLO 训练要读 `data.yaml` + YOLO 格式标注，跟 Mask R-CNN 直接读 COCO 格式的 manifest 不是一回事，这一步只转换一次，转换结果存在本地磁盘（不占 Drive 空间）。

In [7]:
from dataset.convert_coco_to_yolo import convert

YOLO_DATA_DIR = Path('/content/data/yolo')
YOLO_DATA_YAML = YOLO_DATA_DIR / 'data.yaml'

if YOLO_DATA_YAML.exists():
    print('YOLO 格式数据已存在，跳过转换:', YOLO_DATA_YAML)
else:
    yolo_convert_result = convert(CARDD_LOCAL_RAW_DIR, YOLO_DATA_DIR, link_mode='copy')
    print(json.dumps(yolo_convert_result['summary'], indent=2))

print('\ndata.yaml 内容:')
print(YOLO_DATA_YAML.read_text())


{
  "images": {
    "train": 2816,
    "val": 810,
    "test": 374
  },
  "annotations": {
    "train": 6211,
    "val": 1744,
    "test": 785
  },
  "labels": {
    "train": 2816,
    "val": 810,
    "test": 374
  },
  "invalid_polygons": 0
}

data.yaml 内容:
path: /content/data/yolo
train: images/train
val: images/val
test: images/test
names: ["dent", "scratch", "crack", "glass shatter", "lamp broken", "tire flat"]



---
## 2 · 训练 YOLO 分割模型

50 epoch，640 分辨率，seed 42，训练完自动在 test split（不是训练时用的 val split）上评测。

**这一版脚本改了什么、为什么更快（epoch 数完全没变）：**
- `batch=-1`：让 Ultralytics 按显存自动选最大 batch（原来没传，Ultralytics 会退回默认 16，T4 上通常喂不满）。AMP 本身 Ultralytics 默认就是开着的，不用额外设置。
- `cache='ram'`：把解码后的图片缓存进内存，避免每个 epoch 重复解码 JPEG（数据已经在本地磁盘了，这一步是进一步提速）。
- `workers=NUM_WORKERS`：按 Colab 实际给的 CPU 核数来。
- 训练结束后脚本会**显式重新加载 `best.pt`** 再做 test-split 评测，不依赖 Ultralytics 内部是否已经切回最佳权重这种隐性行为——保证下面这行打出来的指标一定是 best checkpoint 的结果。

In [8]:
from training.train_yolo import train as train_yolo

YOLO_OUTPUT_DIR = Path('/content/runs/yolo')

yolo_summary = train_yolo(
    data_yaml=YOLO_DATA_YAML,
    output_dir=YOLO_OUTPUT_DIR,
    epochs=50,
    image_size=640,
    seed=42,
    device=YOLO_DEVICE,
    conf_threshold=0.5,
    iou_match_threshold=0.5,
    model_weights='yolo11n-seg.pt',
    batch=-1,             # 自动选最大能塞进显存的 batch size
    cache='ram',          # 图片解码后缓存进内存，减少重复 IO/解码
    workers=NUM_WORKERS,
)
print(json.dumps(yolo_summary, indent=2))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, forma

In [9]:
# mean matched-pair mask IoU，同 Mask R-CNN 那份 notebook 里对应的一步，YOLO 版本
from evaluation.mask_iou import evaluate as evaluate_yolo_mask_iou

YOLO_BEST_WEIGHTS = YOLO_OUTPUT_DIR / 'yolo_segmentation' / 'weights' / 'best.pt'

yolo_mask_iou_summary = evaluate_yolo_mask_iou(
    manifest_path=MANIFEST_PATH,
    weights_path=YOLO_BEST_WEIGHTS,
    conf_threshold=0.5,
    iou_match_threshold=0.5,
)
print(json.dumps(yolo_mask_iou_summary, indent=2))


{
  "evaluation_split": "test",
  "num_test_images": 374,
  "conf_threshold": 0.5,
  "iou_match_threshold": 0.5,
  "mean_mask_iou": 0.8163,
  "matched_pairs": 425,
  "unmatched_predictions": 121,
  "per_class_mean_mask_iou": {
    "dent": 0.7747,
    "scratch": 0.7522,
    "crack": 0.6555,
    "glass shatter": 0.9461,
    "lamp broken": 0.9289,
    "tire flat": 0.9496
  },
  "error_category_counts": {
    "correct": 425,
    "missed_damage": 360
  },
  "note": "unmatched_predictions are extra detections with no same-class GT instance above iou_match_threshold. classify_error() has no dedicated false-positive category, so it is only applied to matched pairs and to missed GT instances; unmatched_predictions is reported separately instead of being forced into it."
}


In [10]:
from evaluation.merge_results import merge

(YOLO_OUTPUT_DIR / 'training_summary.json').write_text(json.dumps(yolo_summary, indent=2), encoding='utf-8')
yolo_iou_path = RESULTS_DRIVE_DIR / 'yolo_mask_iou_summary.json'
yolo_iou_path.write_text(json.dumps(yolo_mask_iou_summary, indent=2), encoding='utf-8')

yolo_row = merge(
    YOLO_OUTPUT_DIR / 'training_summary.json',
    yolo_iou_path,
    method_label='YOLO Segmentation',
)
(RESULTS_DRIVE_DIR / 'final_results_row_yolo.json').write_text(json.dumps(yolo_row, indent=2), encoding='utf-8')
!cp "{YOLO_OUTPUT_DIR}/training_summary.json" "{RESULTS_DRIVE_DIR}/yolo_training_summary.json"
!cp "{YOLO_BEST_WEIGHTS}" "{RESULTS_DRIVE_DIR}/yolo_best.pt"

print('YOLO 结果已存到 Drive:', RESULTS_DRIVE_DIR)
yolo_row


YOLO 结果已存到 Drive: /content/drive/MyDrive/CarDD_project/colab_results


{'method': 'YOLO Segmentation',
 'mask_iou': 0.8163,
 'precision': 0.8156102275976053,
 'recall': 0.6320886755805253,
 'latency_ms': 13.5,
 'mAP50': 0.604351478514495,
 'mAP50_95': 0.47408559251241233,
 'box_precision': 0.8047254488943553,
 'box_recall': 0.6267195379182832,
 'matched_pairs': 425,
 'unmatched_predictions': 121}

---
训练 + 评测到此结束。结果已经存进 `colab_results/`，可以直接关掉这个 Colab 会话。去 `03_Compare_results.ipynb` 看对比表格（Mask R-CNN 那边没跑完也没关系，会只显示这一行）。